# Muraqqam Challenge — MARBERT v2 Baseline

نفس بنية الـ baseline الرسمي (bert-base-multilingual-cased) لكن بموديل **UBC-NLP/MARBERTv2** المتخصص بالعربي الفصيح.

**الخطوات**
1. تحميل `train.csv` وفصل 10% للتحقق (validation).
2. تحويل كل صف لمسألة تصنيف توكن: تنبؤ برمز من `{0, ., ،, ؟, !, :, ؛, -}` بعد كل كلمة.
3. تدريب `MARBERTv2`.
4. تشغيل الاستدلال على `test.csv` وكتابة `submission.csv` بالصيغة المطلوبة.

> ملاحظة: نفس منطق تقطيع الكلمات المستخدم في مقياس المسابقة بالضبط — هذا يضمن عدم وجود فجوة بين التدريب والتقييم.

### Metric (نفس مقياس المسابقة — نستخدمه محليًا على validation)

In [37]:
"""
Kaggle metric for Arabic Punctuation Restoration.

Conventions:
    - `solution` is the full test CSV, including the hidden gold column.
    - `submission` is the competitor's CSV.
    - Both have a row_id column (already aligned & sorted by Kaggle).
    - `solution` has a column `raw` with the unpunctuated input and a column
      `gold` with the reference punctuated string.
    - `submission` has a column `prediction` with the competitor's
      punctuated string.

Scoring:
    Macro-F1 over the 7 Arabic sentence-punctuation classes
    ( . ، ؟ ! : ؛ - ), EXCLUDING the "no punctuation" class.
"""

import re
from typing import Optional

import pandas as pd
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer

class ParticipantVisibleError(Exception):
    pass

# ---------------------------------------------------------------------------
# Classical Arabic punctuation-restoration set. Anything outside this is
# treated as word content (e.g. parentheses, quotes, numbers) and is NOT
# scored. Competitors must preserve those characters in their predictions.
# ---------------------------------------------------------------------------
VALID_SYMBOLS = set('.،؟!:؛-')


def _tokenize_gold(text: str):
    """
    Split a (gold or prediction) string into (leading_gap, [(word, trailing_gap), ...]).
    A "word" is a maximal run of non-whitespace, non-whitelist chars.
    A "gap" is any run of whitelist chars between words.
    """
    leading_gap_chars = []
    pairs = []
    current_word_chars = []
    in_word = False

    for ch in text:
        if ch.isspace():
            if in_word:
                pairs.append([''.join(current_word_chars), []])
                current_word_chars = []
                in_word = False
            continue

        if ch in VALID_SYMBOLS:
            if in_word:
                pairs.append([''.join(current_word_chars), [ch]])
                current_word_chars = []
                in_word = False
            else:
                if pairs:
                    pairs[-1][1].append(ch)
                else:
                    leading_gap_chars.append(ch)
            continue

        if not in_word:
            in_word = True
            current_word_chars = [ch]
        else:
            current_word_chars.append(ch)

    if in_word:
        pairs.append([''.join(current_word_chars), []])

    return ''.join(leading_gap_chars), [(w, ''.join(g)) for (w, g) in pairs]


def _extract_labels(raw_text: str, generated_text: str, role: str):
    """
    Align `generated_text` to `raw_text` word-by-word and return one list of
    symbols per word, representing the punctuation that appears in the gap
    after each word.

    Raises ValueError on any structural mismatch.
    """
    if raw_text is None or generated_text is None:
        raise ValueError(f"[{role}] text is empty or null")

    raw_words = str(raw_text).strip().split()
    if not raw_words:
        raise ValueError(f"[{role}] raw text contains no words")

    _, pairs = _tokenize_gold(str(generated_text))
    gen_words = [w for (w, _) in pairs]

    if len(gen_words) != len(raw_words):
        raise ValueError(
            f"[{role}] word-count mismatch: raw has {len(raw_words)} words, "
            f"{role} has {len(gen_words)}"
        )

    for i, (rw, gw) in enumerate(zip(raw_words, gen_words)):
        if rw != gw:
            raise ValueError(
                f"[{role}] word mismatch at position {i}: "
                f"raw='{rw}' vs {role}='{gw}'"
            )

    labels = []
    for _, gap in pairs:
        syms = [c for c in gap if c in VALID_SYMBOLS]
        labels.append(syms if syms else ['0'])

    return labels


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    raw_column_name: str = 'text',
    gold_column_name: str = 'final_text',
    prediction_column_name: str = 'final_text',
) -> float:
    """
    Returns macro-F1 over the 7 Arabic punctuation
    classes, excluding the "no punctuation" class.
    """
    solution = solution.copy()
    submission = submission.copy()
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    for col in (raw_column_name, gold_column_name):
        if col not in solution.columns:
            raise RuntimeError(f"Solution is missing column '{col}'")
    if prediction_column_name not in submission.columns:
        raise ParticipantVisibleError(
            f"Submission is missing column '{prediction_column_name}'"
        )

    if len(solution) != len(submission):
        raise ParticipantVisibleError(
            f"Submission has {len(submission)} rows, expected {len(solution)}"
        )

    true_labels = []
    pred_labels = []

    raws = solution[raw_column_name].tolist()
    golds = solution[gold_column_name].tolist()
    preds = submission[prediction_column_name].tolist()

    for idx, (raw, gold, pred) in enumerate(zip(raws, golds, preds)):
        try:
            gold_seq = _extract_labels(raw, gold, role="gold")
        except ValueError as e:
            raise RuntimeError(
                f"Gold extraction failed on row index {idx}: {e}"
            ) from e

        try:
            pred_seq = _extract_labels(raw, pred, role="prediction")
        except ValueError as e:
            raise ParticipantVisibleError(
                f"Prediction at row index {idx} does not align with the raw "
                f"input. Your prediction must contain the same sequence of "
                f"non-punctuation words as the input, with only the allowed "
                f"punctuation symbols {sorted(VALID_SYMBOLS)} inserted "
                f"between them. Details: {e}"
            ) from e

        if len(gold_seq) != len(pred_seq):
            raise ParticipantVisibleError(
                f"Row {idx}: prediction has {len(pred_seq)} word positions "
                f"but the input has {len(gold_seq)}"
            )

        true_labels.extend(gold_seq)
        pred_labels.extend(pred_seq)

    all_classes = sorted(VALID_SYMBOLS) + ['0']
    mlb = MultiLabelBinarizer(classes=all_classes)
    y_true = mlb.fit_transform(true_labels)
    y_pred = mlb.transform(pred_labels)

    classes = list(mlb.classes_)
    zero_idx = classes.index('0')
    scored_cols = [i for i in range(len(classes)) if i != zero_idx]

    return float(f1_score(
        y_true[:, scored_cols],
        y_pred[:, scored_cols],
        average='macro',
        zero_division=0,
    ))

## 1. Setup

In [38]:
import os
import random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
import datasets as hfds

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- MARBERT v2 -------------------------------------------------------------
MODEL_NAME = "UBC-NLP/MARBERTv2"

TRAIN_CSV  = "/kaggle/input/competitions/muraqqamchallenge/train.csv"
TEST_CSV   = "/kaggle/input/competitions/muraqqamchallenge/test.csv"
OUTPUT_DIR = "./marbertv2-punct"
SUBMISSION_CSV = "/kaggle/working/submission.csv"   # لازم working، مو input (input مقروء فقط)

MAX_LEN = 512
BATCH     = 16
EPOCHS    = 40
LR        = 2e-5
GRAD_ACCUM = 1

# The 7 scored punctuation marks plus '0' for "no punctuation"
LABELS    = ["0", ".", "،", "؟", "!", ":", "؛", "-"]
LABEL2ID  = {l: i for i, l in enumerate(LABELS)}
ID2LABEL  = {i: l for i, l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} | Model: {MODEL_NAME}")
if device == "cpu":
    print("WARNING: no GPU detected — training will be much slower.")

Device: cuda | Model: UBC-NLP/MARBERTv2


## 2. Load data

In [39]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")

train_df, val_df = train_test_split(train_df, test_size=0.10, random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print(f"After split: train={len(train_df)} | val={len(val_df)}")

Train rows: 611 | Test rows: 211
After split: train=549 | val=62


## 3. Convert rows into word-level labels

لكل صف نستخدم **نفس منطق تقطيع الكلمات المستخدم في مقياس المسابقة** لنمشي على النص المُرقّم `final_text` ونستخرج قائمتين متوازيتين: الكلمات (تطابق ما سيراه المقياس في `text`) وتسمية الترقيم بعد كل كلمة.

ملاحظات:
- نُقيّم فقط الرموز السبعة. أي شيء آخر (أقواس، تنصيص، أرقام، ﷺ...) يُعامل كجزء من الكلمة.
- الفجوة اللي فيها أكثر من رمز (مثل `؟!`) تُختزل لأول رمز مُقيَّم فيها (تبسيط — نفس الـ baseline الرسمي).

In [40]:
VALID = set("".join(LABELS[1:]))  # the 7 punctuation chars

def _tokenize_with_gaps(text):
    """Walk the string and split into (word, trailing_gap) pairs.
    This matches the competition metric's tokenizer exactly."""
    pairs = []
    cur = []
    in_word = False
    for ch in str(text):
        if ch.isspace():
            if in_word:
                pairs.append([''.join(cur), []])
                cur = []
                in_word = False
            continue
        if ch in VALID:
            if in_word:
                pairs.append([''.join(cur), [ch]])
                cur = []
                in_word = False
            else:
                if pairs:
                    pairs[-1][1].append(ch)
            continue
        if not in_word:
            in_word = True
            cur = [ch]
        else:
            cur.append(ch)
    if in_word:
        pairs.append([''.join(cur), []])
    return [(w, ''.join(g)) for (w, g) in pairs]

def split_to_words_and_labels(final_text):
    """Return (words, label_ids). One label per word."""
    pairs = _tokenize_with_gaps(final_text)
    words = [w for w, _ in pairs]
    labels = []
    for _, gap in pairs:
        # First scored symbol in the gap wins (multi-label collapse).
        chosen = "0"
        for c in gap:
            if c in VALID:
                chosen = c
                break
        labels.append(LABEL2ID[chosen])
    return words, labels

# Sanity check
ex_words, ex_labels = split_to_words_and_labels(train_df["final_text"].iloc[0])
print("First 12 words:", ex_words[:12])
print("First 12 labels:", [i for i in ex_labels[:12]])

First 12 words: ['عانى', 'المسلمون', 'من', 'الجوع', 'والتعب', 'لعدة', 'أيام', 'فقد', 'أنهكتهم', 'أعمال', 'الحفر', 'المتواصلة']
First 12 labels: [0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 1]


## 4. Tokenize and align labels to subwords

MARBERT تستخدم WordPiece مثل BERT، فتُقسّم الكلمات لقطع فرعية (subwords). النهج القياسي: نعطي التسمية لأول قطعة فرعية من كل كلمة، ونتجاهل الباقي (`-100`) عند حساب الخسارة. عند الاستدلال نقرأ التنبؤ من أول قطعة فرعية لكل كلمة.

In [41]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(words_batch, labels_batch=None):
    enc = tokenizer(
        words_batch,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )
    if labels_batch is None:
        return enc
    aligned = []
    for i, labs in enumerate(labels_batch):
        word_ids = enc.word_ids(batch_index=i)
        prev = None
        out = []
        for wid in word_ids:
            if wid is None:
                out.append(-100)
            elif wid != prev:
                out.append(labs[wid])
            else:
                out.append(-100)
            prev = wid
        aligned.append(out)
    enc["labels"] = aligned
    return enc

def build_dataset(df):
    all_words, all_labels = [], []
    CHUNK = MAX_LEN - 2
    for _, row in df.iterrows():
        w, l = split_to_words_and_labels(row["final_text"])
        if not w:
            continue
        for i in range(0, len(w), CHUNK):
            cw = w[i:i + CHUNK]
            cl = l[i:i + CHUNK]
            if cw:
                all_words.append(cw)
                all_labels.append(cl)
    return hfds.Dataset.from_dict({"words": all_words, "labels": all_labels})

## 5. Train

Macro-F1 يُحسب على مستوى الكلمة على الأصناف السبعة (صنف `0` مُستثنى، مطابق لمقياس المسابقة).

In [42]:
import torch.nn as nn
from collections import Counter
from transformers import EarlyStoppingCallback

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    flat_preds, flat_labels = [], []
    for p_row, l_row in zip(preds, labels):
        for p, l in zip(p_row, l_row):
            if l == -100:
                continue
            flat_preds.append(p)
            flat_labels.append(l)
    scored = list(range(1, NUM_LABELS))
    macro_f1 = f1_score(
        flat_labels, flat_preds,
        labels=scored, average="macro", zero_division=0,
    )
    return {"macro_f1": macro_f1}

# class weights: أقل وزن للصنف "0" المهيمن، وزن أعلى للرموز النادرة
flat_train_labels = [l for seq in train_tok["labels"] for l in seq if l != -100]
counts = Counter(flat_train_labels)
total = sum(counts.values())
class_weights = torch.tensor(
    [(total / (NUM_LABELS * counts.get(i, 1))) ** 0.5 for i in range(NUM_LABELS)],
    dtype=torch.float,
)
print("Class weights:", class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device), ignore_index=-100)
        loss = loss_fct(logits.view(-1, NUM_LABELS), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
)
model.to(device)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
)


trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

trainer.train()
print("\nBest validation macro-F1:", trainer.evaluate()["eval_macro_f1"])

Class weights: tensor([0.3914, 1.6370, 1.1764, 5.1044, 5.2696, 2.9597, 3.6229, 2.9930])


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,2.118150,0.012666
2,No log,1.613636,0.000000
3,1.907844,1.246691,0.108443
4,1.907844,0.988844,0.280484
5,1.907844,0.836845,0.374105
6,0.980628,0.732458,0.397084
7,0.980628,0.666430,0.521752
8,0.980628,0.601505,0.567389
9,0.611915,0.582743,0.600898
10,0.611915,0.581356,0.614141


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Best validation macro-F1: 0.6486741125336702


In [43]:
train_df_full = pd.read_csv(TRAIN_CSV)   
full_train_ds = build_dataset(train_df_full)
full_train_tok = full_train_ds.map(tokenize_batch, batched=True, remove_columns=full_train_ds.column_names)

args_final = TrainingArguments(
    output_dir=OUTPUT_DIR + "-final",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=17,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

final_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
)
final_model.to(device)

final_trainer = WeightedTrainer(
    model=final_model,
    args=args_final,
    train_dataset=full_train_tok,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)
final_trainer.train()

model = final_model   

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING

Step,Training Loss
50,1.794471
100,0.955919
150,0.649731
200,0.505369
250,0.427259
300,0.381504
350,0.355836
400,0.340872


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 6. Inference on the test set

لكل صف اختبار نُقطّع الكلمات، نشغّل الموديل، نأخذ التنبؤ من أول قطعة فرعية لكل كلمة، ونعيد دمج الكلمات مع الترقيم المتنبأ به لإنتاج `final_text`.

In [44]:
model.eval()
model.to(device)

@torch.no_grad()
def predict_punctuated(text):
    words = str(text).split()
    if not words:
        return ""
    # Process in chunks to respect MAX_LEN (long docs in this dataset).
    CHUNK = MAX_LEN - 2  # leave room for [CLS] and [SEP]
    preds = []
    i = 0
    while i < len(words):
        chunk = words[i:i + CHUNK]
        enc = tokenizer(
            chunk, is_split_into_words=True, return_tensors="pt",
            truncation=True, max_length=MAX_LEN,
        ).to(device)
        logits = model(**enc).logits[0].cpu().numpy()
        word_ids = enc.word_ids(batch_index=0)
        chunk_preds = [None] * len(chunk)
        for tok_idx, wid in enumerate(word_ids):
            if wid is None or chunk_preds[wid] is not None:
                continue
            chunk_preds[wid] = int(logits[tok_idx].argmax())
        chunk_preds = [p if p is not None else 0 for p in chunk_preds]
        preds.extend(chunk_preds)
        i += CHUNK

    pieces = []
    for w, p in zip(words, preds):
        sym = ID2LABEL[p]
        pieces.append(w if sym == "0" else w + sym)
    return " ".join(pieces)

print(predict_punctuated(test_df["text"].iloc[0])[:200])

كان لدى عبد المطلب وسادة يضعها أمام الكعبة. لم يكن يجلس عليها أحد سواه، لكنه كان يفرح كثيرا لو جلس عليها محمد. وإن حاول أحد إنزاله من عليها؛ قال له: «دعوه فإن لابني هذا شأنا. » كما أن أحدا لم يكن يدخل


## 7. Build submission

In [45]:
import time
from tqdm.auto import tqdm

start = time.time()
preds = [predict_punctuated(t) for t in tqdm(test_df["text"].tolist())]
elapsed = time.time() - start
print(f"Inference time for {len(test_df)} rows: {elapsed/60:.2f} minutes")

submission = pd.DataFrame({"id": test_df["id"], "final_text": preds})
submission.to_csv(SUBMISSION_CSV, index=False)
print(f"Wrote {SUBMISSION_CSV} | {len(submission)} rows")
submission.head()

  0%|          | 0/211 [00:00<?, ?it/s]

Inference time for 211 rows: 0.08 minutes
Wrote /kaggle/working/submission.csv | 211 rows


,id,final_text
0,0,كان لدى عبد المطلب وسادة يضعها أمام الكعبة. لم...
1,1,أحب الصحابة رسول الله- ﷺ- وفضلوه عن أنفسهم، كا...
2,2,أحب الأرقم رسول الله- ﷺ- وكان على استعداد للتض...
3,3,فيلبس ميندل هل الانطباعات الأولى دائمة أم مهمة...
4,4,كلانا غير عقلاني. عبدالله الغذامي يشيع التعبير...


## 8. (اختياري) تحقق محلي من الشكل والدرجة على val

نتأكد إن التسليم يطابق الشكل المطلوب، ونحسب الدرجة محليًا على مجموعة الـ validation باستخدام نفس دالة `score` أعلاه.

In [46]:
val_preds = [predict_punctuated(t) for t in tqdm(val_df["text"].tolist())]
val_sub = pd.DataFrame({"id": val_df["id"] if "id" in val_df.columns else range(len(val_df)),
                          "final_text": val_preds})
val_sol = pd.DataFrame({"id": val_sub["id"], "text": val_df["text"].tolist(),
                          "final_text": val_df["final_text"].tolist()})

local_f1 = score(val_sol.copy(), val_sub.copy(), row_id_column_name="id")
print(f"Local validation macro-F1 (MARBERTv2): {local_f1:.4f}")

  0%|          | 0/62 [00:00<?, ?it/s]

Local validation macro-F1 (MARBERTv2): 0.7497
